In [ ]:
# Instalar las librerías necesarias
%pip install -q ultralytics==8.4.102 matplotlib numpy pandas torch torchvision

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt

import torch

from ultralytics import YOLO

# Semilla para reproducibilidad
SEED = 17
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch:", torch.__version__)
print("GPU disponible:", torch.cuda.is_available())

In [ ]:
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No se detectó GPU. Activa una en Entorno de ejecución → Cambiar tipo de entorno → GPU.")

1.- Se instaló el entorno de trabajo en Google Colab utilizando las librerías PyTorch, Ultralytics YOLO y Matplotlib. Además, se verificó la disponibilidad de una GPU para acelerar el entrenamiento de los modelos de aprendizaje profundo.

In [ ]:
from ultralytics.utils.downloads import download

download("https://github.com/ultralytics/assets/releases/download/v0.0.0/coco8.zip")

In [ ]:
!unzip -q coco8.zip

In [ ]:
import os

print("Contenido del directorio:")
print(os.listdir())

print("\nContenido de coco8:")
print(os.listdir("coco8"))

Se descargó el subconjunto COCO8 para detección y posteriormente COCO8-Seg para segmentación.

In [ ]:
from pathlib import Path

train_images = list(Path("coco8/images/train").glob("*.jpg"))
val_images = list(Path("coco8/images/val").glob("*.jpg"))

print("Imágenes de entrenamiento:", len(train_images))
print("Imágenes de validación:", len(val_images))

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

plt.figure(figsize=(12, 8))

for i, img_path in enumerate(train_images):
    img = Image.open(img_path)

    plt.subplot(2, 2, i + 1)
    plt.imshow(img)
    plt.title(img_path.name)
    plt.axis("off")

plt.tight_layout()
plt.show()

Se verificó el contenido del dataset, contando el número de imágenes disponibles y visualizando algunos ejemplos. Esto permitió confirmar que la descarga fue correcta y conocer las imágenes que serían utilizadas durante el entrenamiento y la evaluación de los modelos.

In [ ]:
from ultralytics import YOLO


model = YOLO("yolov8n.pt")

print(model)

In [ ]:
print("Modelo cargado correctamente.")
print("Tipo:", type(model))

Se cargó el modelo preentrenado YOLOv8n, especializado en detección de objetos. Utilizar un modelo preentrenado reduce considerablemente el tiempo de entrenamiento, ya que parte de conocimientos previamente adquiridos sobre el conjunto de datos COCO.

In [ ]:
results = model.train(
    data="coco8.yaml",
    epochs=10,
    imgsz=640,
    batch=4,
    project="COCO_lab",
    name="yolo_detection"
)

Se entrenó el modelo YOLOv8n utilizando las imágenes de entrenamiento del dataset COCO8. Durante este proceso el modelo ajustó sus parámetros para mejorar la localización y clasificación de los objetos presentes en las imágenes.

In [ ]:
from ultralytics.utils.downloads import download

download("https://github.com/ultralytics/assets/releases/download/v0.0.0/coco8-seg.zip")

In [ ]:
!unzip -q -o coco8-seg.zip

In [ ]:
import os

print(os.listdir("coco8-seg"))

In [ ]:
from ultralytics.utils import ASSETS

print(ASSETS)

In [ ]:
from ultralytics import YOLO


model_seg = YOLO("yolov8n-seg.pt")

print("Modelo de segmentación cargado correctamente.")

In [ ]:
results_seg = model_seg.train(
    data="coco8-seg.yaml",
    epochs=10,
    imgsz=640,
    batch=4,
    project="COCO_lab",
    name="yolo_segmentation"
)

In [ ]:

best_detect = YOLO("/content/runs/detect/COCO_lab/yolo_detection/weights/best.pt")

# Evaluacion
metrics_detect = best_detect.val(data="coco8.yaml")

In [ ]:

best_seg = YOLO("/content/runs/segment/COCO_lab/yolo_segmentation/weights/best.pt")

# Evaluacion
metrics_seg = best_seg.val(data="coco8-seg.yaml")

In [ ]:
from pathlib import Path
from ultralytics import YOLO


best_detect = YOLO("/content/runs/detect/COCO_lab/yolo_detection/weights/best.pt")


test_images = sorted(Path("coco8/images/val").glob("*.jpg"))

# predicciones
results_detect = best_detect.predict(
    source=[str(img) for img in test_images],
    conf=0.25,
    save=True,
    project="COCO_lab",
    name="predicciones_detect",
    exist_ok=True
)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

pred_dir = Path("/content/runs/detect/COCO_lab/predicciones_detect")

pred_images = sorted(pred_dir.glob("*.jpg"))

print(f"Se encontraron {len(pred_images)} imágenes.")

plt.figure(figsize=(12,8))

for i, img_path in enumerate(pred_images):
    img = Image.open(img_path)

    plt.subplot(2, 2, i+1)
    plt.imshow(img)
    plt.title(img_path.name)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
from ultralytics import YOLO

# Cargar el mejor modelo de segmentación
best_seg = YOLO("/content/runs/segment/COCO_lab/yolo_segmentation/weights/best.pt")

# Imágenes de prueba
test_images = sorted(Path("coco8-seg/images/val").glob("*.jpg"))

# Realizar las predicciones
results_seg = best_seg.predict(
    source=[str(img) for img in test_images],
    conf=0.25,
    save=True,
    project="COCO_lab",
    name="predicciones_seg",
    exist_ok=True
)

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

pred_dir = Path("/content/runs/segment/COCO_lab/predicciones_seg")

pred_images = sorted(pred_dir.glob("*.jpg"))

print(f"Se encontraron {len(pred_images)} imágenes.")

plt.figure(figsize=(12,8))

for i, img_path in enumerate(pred_images):
    img = Image.open(img_path)

    plt.subplot(2,2,i+1)
    plt.imshow(img)
    plt.title(img_path.name)
    plt.axis("off")

plt.tight_layout()
plt.show()

Se cargó el modelo YOLOv8n-Seg, diseñado para realizar segmentación de instancias. Posteriormente se entrenó utilizando el conjunto COCO8-Seg, permitiendo al modelo aprender no solo la ubicación de los objetos, sino también la forma exacta de cada uno mediante máscaras de segmentación.

Se realizaron predicciones sobre imágenes del conjunto de validación y se visualizaron los resultados obtenidos. En las imágenes se observaron las cajas delimitadoras, las máscaras de segmentación, las clases detectadas y los niveles de confianza, permitiendo verificar visualmente el funcionamiento de los modelos.

In [ ]:
import numpy as np

ious = []
dices = []

for result in results_seg:

    # Si no hubo máscaras, continuar
    if result.masks is None:
        continue

    pred_masks = result.masks.data.cpu().numpy()

    # Se utiliza la máscara con mayor confianza
    pred_mask = pred_masks[0] > 0.5

    # Cargar la máscara real correspondiente
    gt_mask = result.masks.data.cpu().numpy()[0] > 0.5

    intersection = np.logical_and(pred_mask, gt_mask).sum()
    union = np.logical_or(pred_mask, gt_mask).sum()

    iou = intersection / (union + 1e-8)

    dice = (2 * intersection) / (
        pred_mask.sum() + gt_mask.sum() + 1e-8
    )

    ious.append(iou)
    dices.append(dice)

print(f"IoU promedio : {np.mean(ious):.4f}")
print(f"Dice promedio: {np.mean(dices):.4f}")

In [ ]:
%pip install -q opencv-python

In [ ]:
import cv2
import numpy as np
from pathlib import Path
from ultralytics import YOLO

In [ ]:
from pathlib import Path

label = sorted(Path("coco8-seg/labels/val").glob("*.txt"))[0]

print(label)

with open(label) as f:
    print(f.readline())

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

# Primera imagen del conjunto de validación
image_path = sorted(Path("coco8-seg/images/val").glob("*.jpg"))[0]
label_path = Path("coco8-seg/labels/val") / (image_path.stem + ".txt")

# Leer imagen para conocer su tamaño
image = np.array(Image.open(image_path))
H, W = image.shape[:2]

# Máscara vacía
gt_mask = np.zeros((H, W), dtype=np.uint8)

# Leer todas las anotaciones
with open(label_path) as f:
    lines = f.readlines()

for line in lines:

    data = list(map(float, line.strip().split()))

    # Ignorar la clase
    polygon = np.array(data[1:], dtype=np.float32).reshape(-1, 2)

    # Convertir de coordenadas normalizadas a píxeles
    polygon[:, 0] *= W
    polygon[:, 1] *= H

    polygon = polygon.astype(np.int32)

    cv2.fillPoly(gt_mask, [polygon], 1)

plt.figure(figsize=(6,6))
plt.imshow(gt_mask, cmap="gray")
plt.title("Máscara Ground Truth")
plt.axis("off")
plt.show()

In [ ]:
import cv2
import numpy as np
from pathlib import Path
from PIL import Image

ious = []
dices = []

# Obtener las imágenes de validación
image_paths = sorted(Path("coco8-seg/images/val").glob("*.jpg"))

# Realizar predicciones (sin guardar imágenes)
results = best_seg.predict(
    source=[str(p) for p in image_paths],
    conf=0.25,
    save=False,
    verbose=False
)

for image_path, result in zip(image_paths, results):

    # Leer imagen
    image = np.array(Image.open(image_path))
    H, W = image.shape[:2]

    # ---------------------------
    # Máscara Ground Truth
    # ---------------------------
    gt_mask = np.zeros((H, W), dtype=np.uint8)

    label_path = Path("coco8-seg/labels/val") / (image_path.stem + ".txt")

    with open(label_path) as f:
        lines = f.readlines()

    for line in lines:
        data = list(map(float, line.strip().split()))

        polygon = np.array(data[1:], dtype=np.float32).reshape(-1, 2)
        polygon[:, 0] *= W
        polygon[:, 1] *= H

        polygon = polygon.astype(np.int32)

        cv2.fillPoly(gt_mask, [polygon], 1)

    # ---------------------------
    # Máscara predicha
    # ---------------------------
    pred_mask = np.zeros((H, W), dtype=np.uint8)

    if result.masks is not None:

        masks = result.masks.data.cpu().numpy()

        for mask in masks:

            mask = cv2.resize(
                mask.astype(np.uint8),
                (W, H),
                interpolation=cv2.INTER_NEAREST
            )

            pred_mask = np.maximum(pred_mask, mask)

    # ---------------------------
    # Métricas
    # ---------------------------
    intersection = np.logical_and(gt_mask, pred_mask).sum()
    union = np.logical_or(gt_mask, pred_mask).sum()

    if union > 0:

        iou = intersection / union
        dice = (2 * intersection) / (gt_mask.sum() + pred_mask.sum())

        ious.append(iou)
        dices.append(dice)

print(f"IoU promedio : {np.mean(ious):.4f}")
print(f"Dice promedio: {np.mean(dices):.4f}")


Se construyeron las máscaras reales a partir de las anotaciones del dataset y se compararon con las máscaras predichas por el modelo. Con esta comparación se calcularon el Intersection over Union (IoU) y el Dice Coefficient, métricas ampliamente utilizadas para evaluar la calidad de los modelos de segmentación al medir la similitud entre las máscaras reales y las predichas.

Finalmente se analizaron las métricas obtenidas durante la evaluación. Los resultados mostraron que el modelo de detección alcanzó un buen desempeño en la localización de objetos, mientras que el modelo de segmentación logró identificar adecuadamente las regiones correspondientes a los objetos presentes en las imágenes, obteniendo valores consistentes en Precision, Recall, mAP, IoU y Dice Coefficient. Estos resultados demuestran que ambos modelos fueron entrenados y evaluados correctamente utilizando el conjunto de datos COCO8.